In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader # for image + table+ text kind of. data use --> unstructured.io libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from typing import Annotated, operator
from langchain_core.tools import tool
from tavily import TavilyClient

load_dotenv()

C:\Users\Deepam\AppData\Local\Temp\ipykernel_16512\2662497276.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader # for image + table+ text kind of. data use --> unstructured.io libraries


True

In [10]:
# Import the required modules
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load the pdf files
pdf_loader = PyPDFLoader("../season11/evs_oil_price_shock.pdf")
raw_docs = pdf_loader.load()

# 2. Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = text_splitter.split_documents(raw_docs)
print(f"Number of chunks created: {len(chunks)}")

# 3. Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", 
output_dimensionality=384.
)

# 4. Create the vector store
# The recommended approach is to create the vector store and add documents simultaneously:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_base",
    persist_directory="./chroma_db" # Optional: specify a folder to save the vector store locally
)

# NOTE: If you prefer to initialize it empty and then add documents (like in your original code), 
# you can do it this way:
# vectorstore = Chroma(
#     collection_name="rag_base",
#     embedding_function=embeddings,
#     persist_directory="./chroma_db"
# )
# vectorstore.add_documents(chunks)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Number of chunks created: 51


In [9]:
class AgenticRAGState(MessagesState):
    query: str
    retrieved_docs : Annotated[list[Document], operator.add]
    context : Annotated[str, operator.add]
    generation: str
    need_retrieval : bool

class RouteDecision(BaseModel):
    need_retrieval: bool

In [ ]:
@tool(response_format="content_and_artifact")
def vector_store_search(query: str, k:int = 3):
     """ 
    Search the vector store for relevant document passages.
    Adjust k (default 3) to retrieve more or fewer passages.
    
    """

retriever = vectorstore.as_retriever(search_kwangs={"k": k})
docs = retriever.invoke(query)
context = "\n\n## Vector Store result\n\n " + "\n\n".join(d.page_content for d in docs)

return context,docs

NameError: name 'k' is not defined